# Project Assignment 1

## Setup

In [1]:
master = "spark://spark-master:7077"   # use "spark://spark-master:7077" when running inside the Docker cluster
dataFolder = "/data/project/" if master.startswith("spark://") else "../data/"


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType
from pyspark.sql import Window

spark_builder = (SparkSession.builder
         .master(master)
         .appName("spark-project")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "1536m")   # stay under the worker's --memory 2G cap
         .config("spark.ui.showConsoleProgress", "false"))

if master.startswith("spark://"):
    spark_builder = (spark_builder
                      .config("spark.driver.host", "jupyter")
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 22:33:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://jupyter:4040
Default parallelism (cores seen by the driver): 2


In [3]:
import sys
sys.path.append("/workspace/project")

from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)


## M1

In [6]:
with profiler.profile("M1-baseline"):
    zones_df = (spark.read.option("header", "true")
                .option("inferSchema", "true")
                .csv(dataFolder + "taxi_zone_lookup.csv"))

    raw_df = spark.read.parquet(dataFolder + "yellow_tripdata_*.parquet")

    # Preparar aliases para as zonas para facilitar os joins
    pickup_zones = zones_df.select(
        sf.col("LocationID").alias("pu_loc_id"),
        sf.col("Borough").alias("pickup_borough"),
        sf.col("Zone").alias("pickup_zone")
    )

    dropoff_zones = zones_df.select(
        sf.col("LocationID").alias("do_loc_id"),
        sf.col("Borough").alias("dropoff_borough"),
        sf.col("Zone").alias("dropoff_zone")
    )

    # Base de dados limpa e enriquecida com colunas temporais e zonas já associadas
    base_df = (raw_df.filter(
            (sf.col("fare_amount") > 0) &
            (sf.col("trip_distance") > 0) &
            (sf.col("passenger_count") > 0) &
            (sf.col("passenger_count") <= 6) &
            (sf.col("tpep_dropoff_datetime") > sf.col("tpep_pickup_datetime")) &
            (sf.col("PULocationID").between(1, 263)) &
            (sf.col("DOLocationID").between(1, 263))
        )
        .withColumn("month", sf.month("tpep_pickup_datetime"))
        .withColumn("day_of_week", sf.date_format("tpep_pickup_datetime", "E"))
        .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
        .withColumn("trip_duration_mins", (sf.unix_timestamp("tpep_dropoff_datetime") - sf.unix_timestamp("tpep_pickup_datetime")) / 60.0)
        .join(pickup_zones, sf.col("PULocationID") == sf.col("pu_loc_id"), "inner")
        .join(dropoff_zones, sf.col("DOLocationID") == sf.col("do_loc_id"), "inner")
    )

    base_df.count()


26/11/04 22:38:01 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/project/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: File /data/project/yellow_tripdata_*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.ap

[M1-baseline] wall time: 6.94s (7 stages, 25 tasks) - see profiler.report() / profiler.compare(...)


## M2

In [8]:
with profiler.profile("M2-baseline"):
    (base_df
            .groupBy("pickup_borough", "day_of_week", "hour_of_day")
            .agg(
                sf.count("*").alias("total_trip_count"),
                sf.avg("fare_amount").alias("mean_fare_amount"),
                sf.avg("trip_duration_mins").alias("mean_trip_duration"),
                sf.avg("trip_distance").alias("mean_trip_distance")
            )
            .write.mode("overwrite")
            .format("noop")
            .save()
        )

[M2-baseline] wall time: 11.98s (5 stages, 23 tasks) - see profiler.report() / profiler.compare(...)


## M3

In [19]:
with profiler.profile("M3-baseline"):
    zone_month_stats = (base_df
    .groupBy("pickup_zone", "pickup_borough", "month")
        .agg(
            sf.avg("fare_amount").alias("mean_fare"),
            sf.stddev("fare_amount").alias("stddev_fare"),
            sf.avg("trip_distance").alias("mean_distance"),
            sf.stddev("trip_distance").alias("stddev_distance"),
            sf.count("*").alias("monthly_trip_volume")
        )
    )

    w_borough = Window.partitionBy("pickup_borough", "month")
    w_rolling = Window.partitionBy("pickup_zone").orderBy("month").rowsBetween(-2, 0)

    f = (zone_month_stats
        .withColumn("volume_rank", sf.rank().over(w_borough.orderBy(sf.desc("monthly_trip_volume"))))
        .withColumn("rolling_3m_volume", sf.avg("monthly_trip_volume").over(w_rolling))
        .write.mode("overwrite")
        .format("noop")
        .save()
    )

[M3-baseline] wall time: 11.29s (12 stages, 48 tasks) - see profiler.report() / profiler.compare(...)


## O1

In [13]:
with profiler.profile("O1-baseline"):
    df_o1 = base_df.withColumn("fare_per_km", sf.col("fare_amount") / sf.col("trip_distance"))

    o1_bucket_stats = (df_o1
        .groupBy("PULocationID", "hour_of_day")
        .agg(
            sf.avg("fare_per_km").alias("mean_fpk"),
            sf.stddev("fare_per_km").alias("std_fpk")
        )
    )

    (df_o1.join(o1_bucket_stats, ["PULocationID", "hour_of_day"])
        .withColumn("is_anomaly", sf.col("fare_per_km") > (sf.col("mean_fpk") + 3 * sf.col("std_fpk")))
        .write.mode("overwrite")
       .format("noop")
        .save()
    )

[O1-baseline] wall time: 72.22s (8 stages, 104 tasks) - see profiler.report() / profiler.compare(...)


## O2

In [14]:
with profiler.profile("O2-baseline"):
    od_flows = (base_df
        .groupBy("month", "PULocationID", "DOLocationID", "pickup_borough", "dropoff_borough")
        .agg(sf.count("*").alias("route_trip_count"))
    )

    w_outbound = Window.partitionBy("PULocationID", "month")
    (od_flows
        .withColumn("total_outbound", sf.sum("route_trip_count").over(w_outbound))
        .withColumn("route_share", sf.col("route_trip_count") / sf.col("total_outbound"))
        .write.mode("overwrite")
        .format("noop")
        .save()
    )

[O2-baseline] wall time: 11.81s (7 stages, 37 tasks) - see profiler.report() / profiler.compare(...)


In [ ]:
spark.stop()